In [ ]:
!pip install -q transformers==4.44.0 datasets peft==0.11.1 trl accelerate==0.31.0 bitsandbytes sentencepiece protobuf --no-warn-script-location

In [ ]:
# Try to install flash-attn but don't fail if it doesn't work
try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "flash-attn"], timeout=60, capture_output=True)
    print("Flash Attention 2 installed ")
except:
    print("Flash Attention 2 not available - will use optimized default attention")

In [ ]:
# 2. Verify GPU and Set Up Environment

import torch
import os

# Verify critical imports work
print("Verifying dependencies...")
try:
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    print(" All imports successful")
except ImportError as e:
    print(f" Import error: {e}")
    print("Reinstalling packages...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "--upgrade", "peft", "trl", "accelerate"], capture_output=True)
    print(" Packages upgraded, restart kernel if issues persist")

assert torch.cuda.is_available(), "ERROR: No GPU detected! Please enable GPU in Lightning AI."

# Get GPU info
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU: {gpu_name}")
print(f"VRAM: {gpu_memory:.1f} GB")
print(f"CUDA Version: {torch.version.cuda}")
print(f"PyTorch Version: {torch.__version__}")
print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")

# Set environment variables for optimal A100 performance
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_NO_DEVICE_MAP"] = "1"

# Create output directories
OUTPUT_BASE = "/teamspace/studios/this_studio/outputs"
MODEL_OUTPUT_DIR = f"{OUTPUT_BASE}/mistral_finetuned"
CHECKPOINT_DIR = f"{OUTPUT_BASE}/checkpoints"
FINAL_MODEL_PATH = f"{MODEL_OUTPUT_DIR}/final_adapter"

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\nOutput directory: {MODEL_OUTPUT_DIR}")
print(f"Final model will be saved to: {FINAL_MODEL_PATH}")

In [ ]:
# 3. Upload and Load Dataset

import json
import pandas as pd
from pathlib import Path

# Try multiple possible dataset locations
possible_paths = [
    "/teamspace/studios/this_studio/dataset.jsonl",  # Uploaded to studio root
    "/teamspace/studios/this_studio/data/dataset.jsonl",  # In data folder
    "./dataset.jsonl",  # Current directory
    "../data/dataset.jsonl",
]

dataset_path = None
for path in possible_paths:
    if Path(path).exists():
        dataset_path = path
        break

if dataset_path is None:
    print(" Dataset not found! Please upload dataset.jsonl to one of these locations:")
    for p in possible_paths:
        print(f"   - {p}")
    print("\nThen re-run this cell.")
else:
    print(f" Found dataset at: {dataset_path}")
    
    # Load dataset
    samples = []
    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line or not line.startswith('{'):
                continue
            try:
                samples.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Skipping line {line_num} due to JSON error")
    
    # Convert to DataFrame
    sample_df = pd.DataFrame(samples)
    print(f"Loaded {len(samples)} samples")
    print(f"Columns: {sample_df.columns.tolist()}")
    
    # Auto-detect input/output columns
    input_col = None
    output_col = None
    for col in sample_df.columns:
        col_lower = col.lower()
        if 'prompt' in col_lower or 'desc' in col_lower or 'input' in col_lower or 'instruction' in col_lower:
            input_col = col
        if 'code' in col_lower or 'output' in col_lower or 'response' in col_lower or 'completion' in col_lower:
            output_col = col
    
    # Fallback to first two columns if not detected
    if input_col is None:
        input_col = sample_df.columns[0]
    if output_col is None:
        output_col = sample_df.columns[1] if len(sample_df.columns) > 1 else sample_df.columns[0]
    
    print(f"Input column: {input_col}")
    print(f"Output column: {output_col}")
    print(f"\nSample data preview:")
    print(sample_df.head(2))

In [ ]:
# 4. Preprocess Dataset for Instruction Tuning

from datasets import Dataset

def format_instruction(example):
    """Format data for Mistral instruction tuning."""
    prompt = str(example[input_col]).strip()[:1024]  # Allow longer prompts with A100
    completion = str(example[output_col]).strip()[:2048]  # Allow longer completions
    
    return {
        'prompt': prompt,
        'completion': completion,
        'text': f"<s>[INST] {prompt} [/INST]\n{completion}</s>"
    }

# Create HuggingFace dataset
dataset = Dataset.from_pandas(sample_df)
dataset = dataset.map(format_instruction, remove_columns=sample_df.columns.tolist())

# Split into train/validation (90/10)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
val_dataset = dataset_split['test']

print(f" Training samples: {len(train_dataset)}")
print(f" Validation samples: {len(val_dataset)}")
print(f"\n Sample formatted text (truncated):")
print(train_dataset[0]['text'][:500] + "...")

In [ ]:
# 5. Load Model and Tokenizer 

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Disable auto device mapping to prevent .to() errors with quantized models
os.environ["ACCELERATE_NO_DEVICE_MAP"] = "1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config optimized for A100
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100 has native BF16 support
)

print(f"Loading model (4-bit quantized for efficiency)...")

try:
    # Load model with minimal parameters to avoid device mapping issues
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,  # Load directly to GPU without intermediate CPU copy
    )
    print(" Model loaded successfully!")
except Exception as e:
    print(f" First loading attempt failed: {str(e)[:100]}")
    print("Trying alternative loading method...")
    
    # Fallback: Use device_map=None to avoid dispatch_model issues
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map=None,
    )
    if torch.cuda.is_available():
        model = model.cuda()
    print(" Model loaded with fallback method!")

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

# Print model info
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n Model ready for training!")
print(f" Total parameters: {total_params:,}")
print(f" 4-bit quantization: Enabled")
print(f" Gradient checkpointing: Enabled")
print(f" Device: {'GPU' if next(model.parameters()).is_cuda else 'CPU'}")

In [ ]:
# 6. Configure LoRA and Training (Optimized for A100)

from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import gc

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration - optimized for A100
lora_config = LoraConfig(
    r=32,  # Higher rank for better quality (A100 can handle it)
    lora_alpha=64,  # Scaling factor (typically 2x rank)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Training configuration - optimized for A100 80GB and speed
training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    
    # Batch size settings (larger batches = faster training)
    per_device_train_batch_size=4,  # Larger batch with A100
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    
    # Training duration
    num_train_epochs=3,
    max_steps=300,  # Limit steps to save credits
    
    # Learning rate settings
    learning_rate=2e-4,  # Slightly higher LR for faster convergence
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    
    # Precision (BF16 is optimal for A100)
    bf16=True,
    fp16=False,
    
    # Logging and saving
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    
    # Optimization settings
    optim="paged_adamw_32bit",  # Memory efficient optimizer
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    
    # Data settings
    max_seq_length=2048,  # Longer sequences with A100
    packing=True,  # Pack multiple samples for efficiency
    dataset_text_field="text",  # Our formatted dataset has a 'text' field
    
    # Other settings (CRITICAL - prevents HF token error)
    report_to="none",  # Prevents "huggingface_hub.HfHubHTTPError" when no token
    push_to_hub=False, # Disables uploading to Hub
    seed=42,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Clear memory before training
gc.collect()
torch.cuda.empty_cache()

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    args=training_args,
    tokenizer=tokenizer,
)

# Print training info
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f" LoRA Configuration Complete")
print(f" Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f" Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f" Max steps: {training_args.max_steps}")
print(f" Dataset text field: {training_args.dataset_text_field}")
print(f" Ready to train!")

In [ ]:
# 7. Train the Model

import time
import traceback

def log_gpu_memory():
    """Log current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

print("STARTING TRAINING ON LIGHTNING AI A100")

print(f"\nInitial GPU memory:")
log_gpu_memory()

start_time = time.time()
train_result = None

# Train!
try:
    print("\nTraining in progress... (this may take a while)")
    train_result = trainer.train()
    
    # Calculate training time
    training_time = time.time() - start_time
    minutes = int(training_time // 60)
    seconds = int(training_time % 60)
    
    
    print(" TRAINING COMPLETED SUCCESSFULLY!")
    
    print(f"Training time: {minutes}m {seconds}s")
    print(f"Final training loss: {train_result.training_loss:.4f}")
    log_gpu_memory()
    
except KeyboardInterrupt:
    print("\n\ Training interrupted by user!")
    training_time = time.time() - start_time
    minutes = int(training_time // 60)
    seconds = int(training_time % 60)
    print(f"Completed {minutes}m {seconds}s of training before interruption")
    print("Saving current checkpoint...")
    
except Exception as e:
    print(f"\n Training error encountered!")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {str(e)[:200]}")
    print("\nFull traceback:")
    traceback.print_exc()
    training_time = time.time() - start_time
    minutes = int(training_time // 60)
    seconds = int(training_time % 60)
    print(f"\nCompleted {minutes}m {seconds}s before error")
    print("Attempting to save current state...")
    try:
        trainer.model.save_pretrained(FINAL_MODEL_PATH)
        print(" Partial model state saved")
    except:
        print(" Could not save partial state")

In [ ]:
# 8. Save the Fine-tuned Model

import json

print("Saving fine-tuned model...")

try:
    # Save LoRA adapter
    trainer.model.save_pretrained(FINAL_MODEL_PATH)
    tokenizer.save_pretrained(FINAL_MODEL_PATH)
    print(" Model files saved")
except Exception as e:
    print(f" Error saving model: {e}")
    print("Attempting alternative save method...")
    try:
        model.save_pretrained(FINAL_MODEL_PATH)
        tokenizer.save_pretrained(FINAL_MODEL_PATH)
        print(" Model saved with alternative method")
    except Exception as e2:
        print(f" Could not save model: {e2}")
        raise

# Save training info for reference
training_info = {
    "base_model": MODEL_NAME,
    "final_model_path": FINAL_MODEL_PATH,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "target_modules": list(lora_config.target_modules),  # Convert set to list for JSON serialization
    "training_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
    "max_steps": training_args.max_steps,
    "effective_batch_size": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "final_loss": float(train_result.training_loss) if train_result is not None else None,
}

info_path = f"{FINAL_MODEL_PATH}/training_info.json"
with open(info_path, 'w') as f:
    json.dump(training_info, f, indent=2)


print(" MODEL SAVED SUCCESSFULLY!")

print(f"\nLoRA Adapter saved to: {FINAL_MODEL_PATH}")
print(f"Training info saved to: {info_path}")
print(f"\nTo use this model later, load the adapter with:")
print(f"""
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base_model, "{FINAL_MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained("{FINAL_MODEL_PATH}")
""")

In [ ]:
# 9. Test the Fine-tuned Model (Inference)

def generate_manim_code(prompt, max_new_tokens=512):
    """Generate Manim code from a prompt using the fine-tuned model."""
    model.eval()
    
    input_text = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt")
    
    # Move to GPU AND convert to model's dtype (bfloat16)
    inputs = {k: v.to(model.device).to(torch.bfloat16) if v.dtype != torch.int64 else v.to(model.device) 
              for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract response after [/INST]
    if '[/INST]' in decoded:
        decoded = decoded.split('[/INST]', 1)[-1].strip()
    
    return decoded

# Test prompts
test_prompts = [
    "Create a Manim animation showing the Pythagorean theorem",
    "Write Manim code to visualize a sine wave",
]

print("INFERENCE TEST")

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n--- Test {i} ---")
    print(f"Prompt: {prompt}")
    print(f"\nGenerated Code:")
    try:
        response = generate_manim_code(prompt)
        print(response[:1000] + "..." if len(response) > 1000 else response)
    except Exception as e:
        print(f"Error generating code: {e}")
    print()

In [ ]:
# 10. Download Model Files
# Create a zip file for easy download

import zipfile
import os

zip_filename = f"{OUTPUT_BASE}/mistral_finetuned_adapter.zip"

print("Creating zip file for download...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(FINAL_MODEL_PATH):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, FINAL_MODEL_PATH)
            zipf.write(file_path, arcname)

zip_size = os.path.getsize(zip_filename) / (1024 * 1024)

print("EXPORT COMPLETE!")
print(f"\n Zip file created: {zip_filename}")
print(f"Size: {zip_size:.2f} MB")
print(f"\n Download this file from the Lightning AI file browser.")
print(f"\n Files included:")
for root, dirs, files in os.walk(FINAL_MODEL_PATH):
    for file in files:
        print(f"   - {file}")